# 표현 (Representation)

**과제 내용**

1. **TDM 생성** — 뉴스기사 10개를 전처리(문서별 토큰 저장)하고, 문서별 단어 빈도로 TDM 생성
   - 문서 수가 고정되어 있을 때, TDM의 크기를 줄이기 위한 전처리 방법은?
2. **TF-IDF 계산** — TDM 기반으로 각 단어의 TF-IDF 점수 계산
   - TF-IDF가 TDM보다 유리한 점은? TF와 IDF 각각의 의미는?
3. **N-gram** — 2-gram, 3-gram 생성 후 1·2·3-gram 세 개를 합쳐 TDM 생성
   - N-gram의 크기를 선택할 때 고려할 사항은? n이 커지면 어떤 장단점이 있는가?
4. **단어-문맥 동시등장 행렬** — 기사 하나를 골라 window 크기만큼 이동하며 동시 등장 단어 카운트
   - window_size를 설정할 때 고려할 점은? 크거나 작을 때 각각 어떤 영향이 있는가?

---

**분석 대상 데이터**

- 출처 : 네이버 뉴스 기사 10건 (2026-09-28 ~ 29)
- 카테고리 : 경제 3건, 사회 2건, 생활 2건, IT 3건 (카테고리는 임의 설정, 제목·본문은 페이지를 파싱)

| ID | 카테고리 | 언론사 | 주제 |
|---|---|---|---|
| D00 | 경제 | 연합뉴스TV | 금감원장, 증권사 CEO에 주주환원·사회공헌 확대 주문 |
| D01 | 경제 | 지디넷코리아 | 삼성전기, AI 서버용 MLCC·인덕터 공급 계약 |
| D02 | 경제 | 헤럴드경제 | 서울 원룸 전세보증금·월세 동향 |
| D03 | 사회 | 중앙일보 | 아산 오폐수처리장 남아 시신 발견 |
| D04 | 사회 | KBS | 주택가 흉기 습격 남고생 징역형 구형 |
| D05 | 생활 | 연합뉴스TV | 설악산 첫 단풍, 주 중반 찬 바람 |
| D06 | 생활 | 세계일보 | 췌장암 조기 발견과 증상·검사 |
| D07 | IT | 지디넷코리아 | 앤트로픽 클로드 소넷 5.5 공개 |
| D08 | IT | 경향신문 | 한국 생성형 AI 이용률 (MS 보고서) |
| D09 | IT | 뉴시스 | AI '지능 폭발' 경고, 오픈AI 차세대 모델 보류 |

---
## 0. 환경 설정

- 한국어 형태소 분석 : KoNLPy(Komoran)
- 기사 수집 : requests + BeautifulSoup

In [1]:
%pip install -q konlpy nltk scikit-learn gensim numpy pandas
%pip install -q requests beautifulsoup4 lxml

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import re
import json
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

OUT_DIR = "output"
os.makedirs(OUT_DIR, exist_ok=True)

### 0.1 데이터 : 뉴스 기사 10건

각 기사를 `category / press / title / url / text` 로 정리

- 제목 `#title_area`, 언론사 `a.media_end_head_top_logo img`, 본문 `#dic_area`
- 본문 영역 안에 섞인 기사 외 요소는 태그 단위로 제거한다.
  사진·사진 설명(`span.end_photo_org`, `em.img_desc`), 표로 넣은 사진 설명(`table`),
  요약 부제(`strong.media_end_summary`), 하단 제보 안내(`div.artical-btm`)
- 파서는 `lxml` 을 쓴다. 뉴시스 기사(D09)는 `<b><div>…</b></div>` 처럼 태그가 어긋나 있어서
  `html.parser` 로 읽으면 두 번째 소제목('내부자들의 소신 고발')에서 본문이 끊겨 절반 가까이 사라진다.

In [3]:
# 카테고리는 임의 설정, 제목, 본문은 파싱하여 NEWS에 저장
import requests
from bs4 import BeautifulSoup

ARTICLES = [
    ("경제", "https://n.news.naver.com/mnews/article/422/0000910793"),
    ("경제", "https://n.news.naver.com/mnews/article/092/0002439123"),
    ("경제", "https://n.news.naver.com/mnews/article/016/0002703206"),
    ("사회", "https://n.news.naver.com/mnews/article/025/0003554476"),
    ("사회", "https://n.news.naver.com/mnews/article/056/0012264878"),
    ("생활", "https://n.news.naver.com/mnews/article/422/0000910591"),
    ("생활", "https://n.news.naver.com/article/022/0004162493"),
    ("IT",   "https://n.news.naver.com/mnews/article/092/0002439034"),
    ("IT",   "https://n.news.naver.com/mnews/hotissue/article/032/0003472958?cid=2002206"),
    ("IT",   "https://n.news.naver.com/mnews/article/003/0014220699"),
]

# 본문(#dic_area) 안에 섞여 있는 기사 외 요소
DROP_SELECTORS = ", ".join([
    "span.end_photo_org",        # 사진 + 사진 설명
    "em.img_desc",               # 사진 설명
    "table",                     # 표로 넣은 사진 설명 (세계일보)
    "strong.media_end_summary",  # 요약 부제 (본문 내용 반복)
    "div.artical-btm",           # 하단 제보 안내 (KBS)
])


def parse_article(url):
    """네이버 뉴스 URL -> (언론사, 제목, 본문)"""
    res = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    res.raise_for_status()
    soup = BeautifulSoup(res.text, "lxml")

    press = soup.select_one("a.media_end_head_top_logo img")["alt"]
    title = soup.select_one("#title_area").get_text(strip=True)
    body  = soup.select_one("#dic_area")
    for tag in body.select(DROP_SELECTORS):
        tag.decompose()
    text = re.sub(r"\s+", " ", body.get_text(" ", strip=True))
    return press, title, text


NEWS = []
for category, url in ARTICLES:
    press, title, text = parse_article(url)
    NEWS.append({"category": category, "press": press, "title": title, "url": url, "text": text})

for d in NEWS:
    print(f"[{d['category']:>2}] {d['press']:<7} {len(d['text']):>5}자  {d['title'][:40]}")

[경제] 연합뉴스TV    904자  이찬진 "투자자는 손실, 증권사는 이익…주주환원·사회공헌 확대해야"
[경제] 지디넷코리아    660자  삼성전기, 2900억원 규모 MLCC·인덕터 공급 계약
[경제] 헤럴드경제    1121자  보증금 마련 안그래도 벅찬데…서울 원룸 전세, 한달새 293만원 더 필요
[사회] 중앙일보      482자  아산 오폐수처리장서 20㎝ 남아 시신 발견…경찰 수사 중
[사회] KBS       712자  ‘주택가서 흉기 습격’ 남고생, 징역형 구형
[생활] 연합뉴스TV    915자  설악산 올해 첫 단풍…주 중반부터 찬 바람
[생활] 세계일보     2144자  소화불량인 줄 알았는데…췌장암 43%는 발견 때 이미 전이, 생존율 2.
[IT] 지디넷코리아   2072자  [AI는 지금] 앤트로픽, 오픈AI '가성비 전략'에 맞불…소넷 5.5 
[IT] 경향신문      449자  한국인 10명 중 4명 생성형 AI 이용한다 MS 보고서…1년 새 15%
[IT] 뉴시스      2224자  "10억명 사망 부를 수도"…'지능 폭발' 경고에 오픈AI, 차세대 AI


### 0.2 전처리 파이프라인

`정제 → 형태소 분석(Komoran + 사용자사전) → 품사 필터링 → 표제어 복원 → 표기 통일 → 불용어 제거`

In [4]:
from konlpy.tag import Komoran

# --- 사용자 사전 : 분석기가 쪼개버리는 고유명사/전문용어를 등록 ---
USER_DIC = """\
이찬진\tNNP
금융감독원\tNNP
금융감독원장\tNNP
금융투자협회\tNNP
코스피\tNNP
미래에셋증권\tNNP
주주환원\tNNG
사회공헌\tNNG
자사주\tNNG
배당성향\tNNG
기업가치\tNNG
설명의무\tNNG
단기자금\tNNG
삼성전기\tNNP
적층세라믹커패시터\tNNG
커패시터\tNNG
인덕터\tNNG
장기공급계약\tNNG
데이터센터\tNNG
탑재량\tNNG
수동부품\tNNG
고객사\tNNG
전세보증금\tNNG
월세보증금\tNNG
다방\tNNP
다방여지도\tNNP
국토교통부\tNNP
자치구\tNNG
전용면적\tNNG
전월세\tNNG
대학가\tNNG
관리비\tNNG
품귀\tNNG
삼성디스플레이\tNNP
아산경찰서\tNNP
아산소방서\tNNP
그린센터\tNNP
오폐수처리장\tNNG
오폐수\tNNG
집수시설\tNNG
남아\tNNG
유산아\tNNG
사생아\tNNG
압수영장\tNNG
서울남부지법\tNNP
스토킹처벌법\tNNP
여고생\tNNG
남고생\tNNG
부장판사\tNNG
살인미수\tNNG
징역형\tNNG
피고인\tNNG
죗값\tNNG
설악산\tNNP
오대산\tNNP
북한산\tNNP
기상청\tNNP
산\tNNG
단풍나무\tNNG
예보분석관\tNNG
일교차\tNNG
최고조\tNNG
보건복지부\tNNP
중앙암등록본부\tNNP
국가암등록통계\tNNP
국가암정보센터\tNNP
췌장암\tNNG
췌장\tNNG
진단자\tNNG
소화불량\tNNG
원격전이\tNNG
상대생존율\tNNG
생존율\tNNG
장기\tNNG
복부초음파\tNNG
내시경초음파\tNNG
담관\tNNG
담도염\tNNG
담도질환\tNNG
소화기내과\tNNG
종양표지자\tNNG
조기검진\tNNG
근치적\tNNG
항암치료\tNNG
앤트로픽\tNNP
오픈AI\tNNP
클로드\tNNP
소넷\tNNP
오퍼스\tNNP
아스트라\tNNP
솔\tNNP
터미널벤치\tNNP
차토그래피\tNNP
프런티어코드\tNNP
아마존웹서비스\tNNP
마이크로소프트\tNNP
가성비\tNNG
코딩\tNNG
사이버보안\tNNG
안전장치\tNNG
분류기\tNNG
테스터\tNNG
이코노미\tNNG
인스티튜트\tNNG
생성형\tNNG
생산연령인구\tNNG
이용\tNNG
이용률\tNNG
순위\tNNG
아랍에미리트연합\tNNP
케임브리지대\tNNP
토론토대\tNNP
몬트리올대\tNNP
야쿠프\tNNP
파호츠키\tNNP
힌턴\tNNP
요슈아\tNNP
벤지오\tNNP
제이컵\tNNP
콕슨\tNNP
허빙어\tNNP
앨버니지\tNNP
올트먼\tNNP
메디케어\tNNP
데브데이\tNNP
정렬과학팀\tNNP
개발진\tNNG
선순환\tNNG
초지능\tNNG
내부자\tNNG
"""

DIC_PATH = os.path.join(OUT_DIR, "user_dic.txt")
with open(DIC_PATH, "w", encoding="utf-8") as f:
    f.write(USER_DIC)

komoran = Komoran(userdic=DIC_PATH)
print(komoran.pos("앤트로픽 개발진이 클로드 소넷을 공개했다."))

[('앤트로픽', 'NNP'), ('개발진', 'NNG'), ('이', 'JKS'), ('클로드', 'NNP'), ('소넷', 'NNP'), ('을', 'JKO'), ('공개', 'NNG'), ('하', 'XSV'), ('았', 'EP'), ('다', 'EF'), ('.', 'SF')]


In [5]:
# 분석에 남길 품사만 지정
KEEP_POS  = {"NNG", "NNP", "VV", "VA", "MAG", "SL"}
PREDICATE = {"VV", "VA"}  # 어간 + '다' 로 표제어 복원

STOP_WORDS = set("""
것 등 수 때 중 점 내 외 위 뒤 앞 안 밖 곳 만큼 가지 대비 기준 경우 관련 통해 대해 위해 이상 이하
하다 되다 있다 없다 이다 아니다 같다 보다 두다 오다 가다 내다 지다 늘다 낮다 높다 크다 많다 적다
위하다 대하다 통하다 따르다 나타나다 이어지다 밝히다 전하다 설명하다 강조하다 기록하다
지나다 알리다 나서다 나오다 받다 들다 서다 그치다
이번 지난 최근 올해 내년 현재 향후 이후 이전 지난해 전년 동기 기간 오늘 전날 이날 다음 가운데
우리 그것 이것 이런 그런 이어 다만 특히 또한 함께 가장 매우 이미 실제 대한 각각
기자 사진 제공 자료 출처 보도 외신 업계 회사 관계자 밝혀 소개 공개 발표 진행 확대 강화
기록 강조 설명 당부 주장 해달
""".split())

KEEP_ONE_CHAR = {"칩", "팀", "차", "웹", "앱", "폰", "법", "암", "산", "솔"}

# 같은 대상을 가리키는 다른 표기를 하나로 통일
SYNONYMS = {"인공지능": "AI", "금감원": "금융감독원", "MS": "마이크로소프트", "CEO": "최고경영자"}

# 숫자 + 단위 (293만원, 43.3%, 15도, 20㎝ ...)
#   문서 구분에 도움이 되지 않고, '15도로 -> 도로' 같은 오분석을 만든다
NUM_UNIT = r"\d+(?:[.,]\d+)*(?:%포인트|%P|%|천억원|억원|만원|원|도|명|건|개사|개|배|세|위|점|㎝|㎞|g)?"


def clean_text(text):
    """분석에 방해되는 요소 제거 (이메일, URL, 말머리, 기자명, 제보 안내, 숫자, 특수문자)"""
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"\[[^\]]*\]|<[^>]*>", " ", text)                        # [헤럴드경제=OOO 기자], [앵커], <발언자>
    text = re.sub(r"\((?:사진|표)=[^)]*\)", " ", text)                      # (사진=OOO)
    text = re.sub(r"[가-힣]{2,4} 기자 ?=", " ", text)                        # OOO 기자 =
    text = re.sub(r"연합뉴스TV [가-힣]{2,4}입니다\.|[가-힣]{2,4} 기자입니다\.", " ", text)  # 방송 기사 클로징 멘트
    text = re.sub(r"연합뉴스TV 기사문의.*$", " ", text)                      # 하단 제보 안내
    text = re.sub(r"#\S+", " ", text)                                       # 해시태그
    text = re.sub(r"\((?:[A-Z]{2,}|R&D)\)", "", text)                       # 인공지능(AI) : 앞말을 반복하는 영문 약어
    text = text.replace("R&D", "연구개발")
    text = re.sub(NUM_UNIT, " ", text)
    text = re.sub(r"[^가-힣A-Za-z0-9\s\.\,\?\!\'\"%\-\(\)]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def preprocess(text):
    """문자열 -> 분석용 토큰 리스트"""
    tokens = []
    for word, pos in komoran.pos(clean_text(text)):
        if pos not in KEEP_POS:
            continue
        if pos in PREDICATE:
            word = word + "다"
        elif pos == "SL":
            word = word.upper()
        word = SYNONYMS.get(word, word)
        if word in STOP_WORDS:
            continue
        if len(word) == 1 and word not in KEEP_ONE_CHAR:
            continue
        # Komoran 사전에는 '중부 지방'처럼 공백을 포함한 고유명사가 있다.
        # 한 단어로써 처리할 때에는 공백 대신 '_' 로 바꿔둔다.
        tokens.append(word.replace(" ", "_"))
    return tokens


print(preprocess("[서울=뉴시스]홍길동 기자 = 인공지능(AI) 모델이 15도로 오른 기온을 예측했다."))

['AI', '모델', '오르다', '기온', '예측']


---
## 1. TDM (Term-Document Matrix) 생성

### 1.1 문서별 전처리 및 토큰 저장

In [6]:
doc_ids    = [f'D{str(i).zfill(2)}' for i in range(len(NEWS))]
doc_titles = [d["title"] for d in NEWS]
doc_cats   = [d["category"] for d in NEWS]
doc_ls     = [preprocess(d["text"]) for d in NEWS]      # ★ 문서별 토큰 리스트

for i, tokens in enumerate(doc_ls):
    print(f"{doc_ids[i]} [{doc_cats[i]:>2}] 토큰 {len(tokens):>4}개 / 고유 {len(set(tokens)):>3}개"
          f" | {doc_titles[i][:30]}")
    print(f"        {tokens[:12]}")

D00 [경제] 토큰  126개 / 고유 104개 | 이찬진 "투자자는 손실, 증권사는 이익…주주환원·사회공
        ['이찬진', '금융감독원장', '증권사', '최고경영자', '만나다', '주주환원', '사회공헌', '투자자', '보호', '힘쓰다', '주문', '원장']
D01 [경제] 토큰  103개 / 고유  65개 | 삼성전기, 2900억원 규모 MLCC·인덕터 공급 계약
        ['삼성전기', '글로벌', '대형', '기업', '규모', 'AI', '서버', '적층세라믹커패시터', '인덕터', '공급', '계약', '체결']
D02 [경제] 토큰  159개 / 고유  73개 | 보증금 마련 안그래도 벅찬데…서울 원룸 전세, 한달새 
        ['서울', '연립', '다세대', '원룸', '평균', '전세보증금', '전월', '오르다', '평균', '월세', '자치구', '용산구']
D03 [사회] 토큰   79개 / 고유  56개 | 아산 오폐수처리장서 20㎝ 남아 시신 발견…경찰 수사 
        ['충남', '아산시', '삼성디스플레이', '사업장', '오폐수처리장', '남아', '시신', '발견', '경찰', '수사', '아산경찰서', '아산소방서']
D04 [사회] 토큰  106개 / 고유  86개 | ‘주택가서 흉기 습격’ 남고생, 징역형 구형
        ['서울', '주택가', '여고생', '흉기', '휘두르다', '남자', '고등학생', '검찰', '징역형', '구형', '서울남부지법', '형사']
D05 [생활] 토큰  116개 / 고유  82개 | 설악산 올해 첫 단풍…주 중반부터 찬 바람
        ['추석', '연휴', '끝나다', '설악산', '단풍', '물들다', '날씨', '중반', '차다', '바람', '불어오다', '서울']
D06 [생활] 토큰  350개 / 고유 175개 | 소화불량인 줄 알았는데…췌장암 43%는 발견 때 이미 
        ['췌장암', '진단자', '발견', '당시'

In [7]:
# 문서별 처리된 토큰을 파일로 저장
with open(f'{OUT_DIR}/doc_tokens.json', "w", encoding="utf-8") as f:
    json.dump(
        [
            {"id": doc_ids[i], "title": doc_titles[i], "category": doc_cats[i], "tokens": doc_ls[i]}
            for i in range(len(NEWS))],
        f, ensure_ascii=False, indent=2)
print(f"-> {OUT_DIR}/doc_tokens.json 저장 완료")

-> output/doc_tokens.json 저장 완료


### 1.2 직접 구현

`{단어 : 인덱스}` 사전을 만들고 (단어 x 문서) 행렬에 빈도를 채워 넣는다.

In [8]:
word2id = defaultdict(lambda: len(word2id))
[word2id[token] for doc in doc_ls for token in doc]      # 등장 순서대로 인덱스 부여

TDM = np.zeros((len(word2id), len(doc_ls)), dtype=int)   # (단어 x 문서)
for j, doc in enumerate(doc_ls):
    for token in doc:
        TDM[word2id[token], j] += 1

sorted_vocab = sorted((idx, w) for w, idx in word2id.items())
vocab = [w for _, w in sorted_vocab]

df_TDM = pd.DataFrame(TDM, index=vocab, columns=doc_ids)
df_TDM.index.name = "단어"

print("TDM shape (단어 x 문서) :", df_TDM.shape)
df_TDM.loc[df_TDM.sum(axis=1).sort_values(ascending=False).index[:15]]

TDM shape (단어 x 문서) : (917, 10)


,D00,D01,D02,D03,D04,D05,D06,D07,D08,D09
단어,,,,,,,,,,
AI,0,5,0,0,0,0,0,4,4,35
소넷,0,0,0,0,0,0,0,30,0,0
평균,0,0,21,0,0,0,0,0,1,0
췌장암,0,0,0,0,0,0,20,0,0,0
모델,0,0,0,0,0,0,0,14,0,4
작업,0,0,0,0,0,0,0,11,0,2
서울,0,0,9,0,2,2,0,0,0,0
오픈AI,0,0,0,0,0,0,0,6,0,7
성능,0,1,0,0,0,0,0,9,0,3


### 1.3 sklearn CountVectorizer 활용

문서는 토큰화한 결과를 공백으로 이어붙여 전달한다.

- `lowercase=False` : `AI` → `ai` 변환 금지
- `token_pattern=r"\S+"` : 기본값 `(?u)\b\w\w+\b` 는 두 글자 이상만 토큰으로 인정해서
  전처리에서 일부러 남긴 한 글자 토큰(`암`, `산`, `솔`)을 버린다. 이미 나눠 둔 공백 기준으로만 자른다.

In [9]:
from sklearn.feature_extraction.text import CountVectorizer

docs_joined = [" ".join(tokens) for tokens in doc_ls]

count_vect = CountVectorizer(lowercase=False, token_pattern=r"\S+")
DTM = count_vect.fit_transform(docs_joined)              # (문서 x 단어)

sk_vocab = count_vect.get_feature_names_out()
df_TDM_sk = pd.DataFrame(DTM.toarray().T, index=sk_vocab, columns=doc_ids)
df_TDM_sk.index.name = "단어"

print("DTM shape (문서 x 단어) :", DTM.shape)
print("직접 구현 어휘 수 :", len(vocab), " / sklearn 어휘 수 :", len(sk_vocab))
print("직접 구현과 값이 동일 :", df_TDM_sk.sort_index().equals(df_TDM.sort_index()))

# 기본 token_pattern 을 썼다면 사라졌을 토큰
default_vocab = CountVectorizer(lowercase=False).fit(docs_joined).get_feature_names_out()
print("기본 token_pattern 이 버리는 토큰 :", sorted(set(sk_vocab) - set(default_vocab)))

check = ["AI", "오픈AI", "서울", "췌장암", "투자자"]
df_TDM_sk.loc[check]

DTM shape (문서 x 단어) : (10, 917)
직접 구현 어휘 수 : 917  / sklearn 어휘 수 : 917
직접 구현과 값이 동일 : True
기본 token_pattern 이 버리는 토큰 : ['산', '솔', '암']


,D00,D01,D02,D03,D04,D05,D06,D07,D08,D09
단어,,,,,,,,,,
AI,0,5,0,0,0,0,0,4,4,35
오픈AI,0,0,0,0,0,0,0,6,0,7
서울,0,0,9,0,2,2,0,0,0,0
췌장암,0,0,0,0,0,0,20,0,0,0
투자자,8,0,0,0,0,0,0,0,0,0


### 1.4 TDM 의 크기와 희소성(sparsity)

In [10]:
from scipy import sparse

n_terms, n_docs = TDM.shape
nnz = int(np.count_nonzero(TDM))
print(f"행렬 크기       : {n_terms} x {n_docs} = {n_terms * n_docs:,} 칸")
print(f"0이 아닌 원소   : {nnz:,} 개")
print(f"희소도(sparsity): {(1 - nnz / (n_terms * n_docs)) * 100:.2f} %  <- 대부분이 0")

# 밀집 배열 vs 희소 행렬(CSR : 0이 아닌 값 + 열 인덱스 + 행 시작 위치만 저장)
TDM_csr = sparse.csr_matrix(TDM)
dense_bytes  = TDM.nbytes
sparse_bytes = TDM_csr.data.nbytes + TDM_csr.indices.nbytes + TDM_csr.indptr.nbytes
print(f"\n밀집 배열 메모리 : {dense_bytes:,} bytes")
print(f"CSR 희소 행렬    : {sparse_bytes:,} bytes  ({(1 - sparse_bytes / dense_bytes) * 100:.1f}% 절약)")

행렬 크기       : 917 x 10 = 9,170 칸
0이 아닌 원소   : 1,076 개
희소도(sparsity): 88.27 %  <- 대부분이 0

밀집 배열 메모리 : 73,360 bytes
CSR 희소 행렬    : 16,584 bytes  (77.4% 절약)


### 1.5 [실험] 전처리 단계별 어휘 수 변화

문서 수(열)는 10개로 고정이므로 TDM 의 크기는 **어휘 수(행)** 가 결정한다.
전처리 단계를 하나씩 쌓아 가며 10개 문서 전체의 어휘 수가 어떻게 줄어드는지 확인한다.

In [11]:
def stage_tokens(text):
    """전처리 단계별 토큰 리스트 (각 단계는 앞 단계를 포함)"""
    cleaned = clean_text(text)
    morphs  = komoran.pos(cleaned)

    s = {}
    s["① 원문 공백 분리"]           = text.split()
    s["② 정제 후 공백 분리"]         = cleaned.split()
    s["③ 형태소 분석"]               = [w for w, _ in morphs]
    s["④ + 품사 필터·표제어 복원"]   = [w + "다" if p in PREDICATE else (w.upper() if p == "SL" else w)
                                       for w, p in morphs if p in KEEP_POS]
    s["⑤ + 표기 통일"]               = [SYNONYMS.get(w, w) for w in s["④ + 품사 필터·표제어 복원"]]
    s["⑥ + 불용어·한 글자 제거"]     = preprocess(text)
    return s


stages = [stage_tokens(d["text"]) for d in NEWS]

rows = []
for name in stages[0]:
    rows.append({"단계": name, "어휘 수": len({w for s in stages for w in s[name]})})

# 문서 빈도 기반 필터 : 2개 이상 문서에 등장한 단어만
cv_min2 = CountVectorizer(lowercase=False, token_pattern=r"\S+", min_df=2).fit(docs_joined)
rows.append({"단계": "⑦ + 2개 이상 문서에 등장 (min_df=2)", "어휘 수": len(cv_min2.vocabulary_)})

df_stage = pd.DataFrame(rows)
df_stage["TDM 크기"]      = df_stage["어휘 수"].map(lambda v: f"{v} x {len(NEWS)}")
df_stage["직전 대비(%)"]  = (df_stage["어휘 수"].pct_change() * 100).round(1)
df_stage["원문 대비(%)"]  = (df_stage["어휘 수"] / df_stage["어휘 수"][0] * 100).round(1)
df_stage

,단계,어휘 수,TDM 크기,직전 대비(%),원문 대비(%)
0,① 원문 공백 분리,1888,1888 x 10,NaN,100.0
1,② 정제 후 공백 분리,1741,1741 x 10,-7.8,92.2
2,③ 형태소 분석,1198,1198 x 10,-31.2,63.5
3,④ + 품사 필터·표제어 복원,1043,1043 x 10,-12.9,55.2
4,⑤ + 표기 통일,1041,1041 x 10,-0.2,55.1
5,⑥ + 불용어·한 글자 제거,917,917 x 10,-11.9,48.6
6,⑦ + 2개 이상 문서에 등장 (min_df=2),130,130 x 10,-85.8,6.9


In [12]:
# min_df=2 로 사라지는 단어 중 빈도가 높은 단어 : 대부분 한 문서의 주제어다
removed = df_TDM.index.difference(cv_min2.get_feature_names_out())
top_removed = df_TDM.loc[removed].sum(axis=1).nlargest(8)
print(f"min_df=2 로 제거되는 단어 {len(removed)}개 중 빈도 상위")
print("  " + ", ".join(f"{w}({c}회, {df_TDM.loc[w].idxmax()})" for w, c in top_removed.items()))

min_df=2 로 제거되는 단어 787개 중 빈도 상위
  소넷(30회, D07), 췌장암(20회, D06), 월세(12회, D02), 개발(8회, D09), 단풍(8회, D05), 췌장(8회, D06), 클로드(8회, D07), 투자자(8회, D00)


In [13]:
# 형태소 분석 전 : 같은 단어가 조사·어미에 따라 서로 다른 열이 된다
eojeols = sorted({w for s in stages for w in s["② 정제 후 공백 분리"]})
for stem in ["췌장암", "투자자", "앤트로픽"]:
    print(f"{stem:<5} : {[w for w in eojeols if w.startswith(stem)]}")

췌장암   : ['췌장암', '췌장암과', '췌장암보다', '췌장암에서', '췌장암에서는', '췌장암은', '췌장암을', '췌장암의', '췌장암이']
투자자   : ['투자자', '투자자가']
앤트로픽  : ['앤트로픽', '앤트로픽에서', '앤트로픽은', '앤트로픽의', '앤트로픽이']


---
# 2. TF-IDF 계산

- **TF** (Term Frequency) = 문서 내 단어 빈도 / 문서 내 전체 단어 수
- **IDF** (Inverse Document Frequency) = ln(전체 문서 수 / 그 단어가 등장한 문서 수)
- **TF-IDF** = TF x IDF

## 2.1 직접 계산

In [14]:
tf       = df_TDM / df_TDM.sum(axis=0)                  # 문서(열)별 합이 1
doc_freq = (df_TDM > 0).sum(axis=1)                     # 단어별 등장 문서 수 (DF)
idf      = np.log(len(doc_ids) / doc_freq)
tf_idf   = tf.mul(idf, axis=0)                          # (단어 x 문서)

print("TF-IDF shape (단어 x 문서) :", tf_idf.shape)
tf_idf.loc[df_TDM.sum(axis=1).sort_values(ascending=False).index[:8]].round(3)

TF-IDF shape (단어 x 문서) : (917, 10)


,D00,D01,D02,D03,D04,D05,D06,D07,D08,D09
단어,,,,,,,,,,
AI,0.0,0.044,0.000,0.0,0.000,0.000,0.000,0.011,0.068,0.080
소넷,0.0,0.000,0.000,0.0,0.000,0.000,0.000,0.207,0.000,0.000
평균,0.0,0.000,0.213,0.0,0.000,0.000,0.000,0.000,0.030,0.000
췌장암,0.0,0.000,0.000,0.0,0.000,0.000,0.132,0.000,0.000,0.000
모델,0.0,0.000,0.000,0.0,0.000,0.000,0.000,0.068,0.000,0.016
작업,0.0,0.000,0.000,0.0,0.000,0.000,0.000,0.053,0.000,0.008
서울,0.0,0.000,0.068,0.0,0.023,0.021,0.000,0.000,0.000,0.000
오픈AI,0.0,0.000,0.000,0.0,0.000,0.000,0.000,0.029,0.000,0.028


In [15]:
# IDF 값 비교 : 여러 문서에 나올수록 IDF 가 작아진다
df_idf = pd.DataFrame({"등장 문서 수(DF)": doc_freq,
                       "IDF": idf.round(3),
                       "총 빈도": df_TDM.sum(axis=1)})

print("[여러 문서에 두루 등장 -> IDF 낮음]")
display(df_idf.sort_values(["등장 문서 수(DF)", "총 빈도"], ascending=[False, False]).head(8))
print("\n[한 문서에만 등장 -> IDF 최대]")
display(df_idf[df_idf["등장 문서 수(DF)"] == 1].sort_values("총 빈도", ascending=False).head(5))

[여러 문서에 두루 등장 -> IDF 낮음]


,등장 문서 수(DF),IDF,총 빈도
단어,,,
빠르다,5,0.693,11
AI,4,0.916,48
상태,4,0.916,5
보이다,4,0.916,4
성능,3,1.204,13
서울,3,1.204,13
필요,3,1.204,11
속도,3,1.204,8



[한 문서에만 등장 -> IDF 최대]


,등장 문서 수(DF),IDF,총 빈도
단어,,,
소넷,1,2.303,30
췌장암,1,2.303,20
월세,1,2.303,12
개발,1,2.303,8
투자자,1,2.303,8


## 2.2 sklearn TfidfVectorizer 활용

sklearn 은 스무딩된 IDF( `ln((1+n)/(1+df)) + 1` )와 L2 정규화를 기본 적용하므로
값의 절대 크기는 직접 계산과 다르다. 문서별 상위 5개 단어가 얼마나 겹치는지 확인한다.

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vect = TfidfVectorizer(lowercase=False, token_pattern=r"\S+")
tfidf_sk = tfidf_vect.fit_transform(docs_joined)
sk_terms = tfidf_vect.get_feature_names_out()

df_tfidf_sk = pd.DataFrame(tfidf_sk.toarray().T, index=sk_terms, columns=doc_ids)
print("sklearn TF-IDF shape (문서 x 단어) :", tfidf_sk.shape)

rows = []
for did in doc_ids:
    mine = tf_idf[did].nlargest(5).index.tolist()
    sk   = df_tfidf_sk[did].nlargest(5).index.tolist()
    rows.append({"문서": did, "직접 계산 top5": ", ".join(mine), "sklearn top5": ", ".join(sk),
                 "겹치는 수": len(set(mine) & set(sk))})
pd.DataFrame(rows)

sklearn TF-IDF shape (문서 x 단어) : (10, 917)


,문서,직접 계산 top5,sklearn top5,겹치는 수
0,D00,"투자자, 증권사, 사회공헌, 주주환원, 보호","투자자, 증권사, 사회공헌, 금융감독원, 눈높이",3
1,D01,"삼성전기, MLCC, 서버, 공급, 계약","MLCC, 삼성전기, 계약, 공급, 서버",5
2,D02,"평균, 월세, 원룸, 전세보증금, 자치구","평균, 월세, 원룸, 전세보증금, 서울",4
3,D03,"시신, 사업장, 경찰, 아산시, 삼성디스플레이","시신, 경찰, 사업장, 남아, 삼성디스플레이",4
4,D04,"피해자, 범행, 주택가, 흉기, 휘두르다","피해자, 범행, 검찰, 고등학생, 고려",2
5,D05,"단풍, 물들다, 기온, 설악산, 중부_지방","단풍, 기온, 물들다, 산, 설악산",4
6,D06,"췌장암, 황달, 췌장, 수술, 체중","췌장암, 췌장, 황달, 수술, 증상",4
7,D07,"소넷, 모델, 클로드, 평가, 작업","소넷, 모델, 작업, 클로드, 평가",5
8,D08,"분기, 한국, 생성형, 세계, AI","AI, 분기, 생성형, 세계, 한국",5
9,D09,"AI, 개발, 경고, 연구개발, 발전","AI, 개발, 경고, 연구개발, 오픈AI",4


## 2.3 [비교] 단순 빈도 상위어 vs TF-IDF 상위어

In [17]:
for i, did in enumerate(doc_ids):
    top_tf    = df_TDM[did].sort_values(ascending=False).head(5).index.tolist()
    top_tfidf = tf_idf[did].sort_values(ascending=False).head(5).index.tolist()
    print(f"\n{did} [{doc_cats[i]}] {doc_titles[i][:34]}")
    print(f"   빈도   : {top_tf}")
    print(f"   TF-IDF : {top_tfidf}")


D00 [경제] 이찬진 "투자자는 손실, 증권사는 이익…주주환원·사회공헌 확대
   빈도   : ['투자자', '증권사', '사회공헌', '손실', '주문']
   TF-IDF : ['투자자', '증권사', '사회공헌', '손실', '주문']

D01 [경제] 삼성전기, 2900억원 규모 MLCC·인덕터 공급 계약
   빈도   : ['삼성전기', 'MLCC', 'AI', '서버', '공급']
   TF-IDF : ['MLCC', '삼성전기', '공급', '서버', '계약']

D02 [경제] 보증금 마련 안그래도 벅찬데…서울 원룸 전세, 한달새 293만
   빈도   : ['평균', '월세', '서울', '원룸', '전세보증금']
   TF-IDF : ['평균', '월세', '원룸', '전세보증금', '자치구']

D03 [사회] 아산 오폐수처리장서 20㎝ 남아 시신 발견…경찰 수사 중
   빈도   : ['시신', '사업장', '경찰', '삼성디스플레이', '오폐수처리장']
   TF-IDF : ['시신', '경찰', '사업장', '삼성디스플레이', '아산시']

D04 [사회] ‘주택가서 흉기 습격’ 남고생, 징역형 구형
   빈도   : ['피해자', '범행', '다치다', '흉기', '주택가']
   TF-IDF : ['피해자', '범행', '징역형', '다치다', '검찰']

D05 [생활] 설악산 올해 첫 단풍…주 중반부터 찬 바람
   빈도   : ['단풍', '물들다', '기온', '산', '설악산']
   TF-IDF : ['단풍', '물들다', '기온', '산', '설악산']

D06 [생활] 소화불량인 줄 알았는데…췌장암 43%는 발견 때 이미 전이, 
   빈도   : ['췌장암', '필요', '췌장', '황달', '수술']
   TF-IDF : ['췌장암', '췌장', '황달', '수술', '증상']

D07 [IT] [AI는 지금] 앤트로픽, 오픈AI '가성비 전략'에 맞불…소
   빈도   : ['소넷', '모델', '작업', '

## 2.4 [실험] 문서 유사도 : TDM vs TF-IDF

두 표현으로 문서 간 코사인 유사도를 구하고, 각 문서와 **가장 비슷한 문서**가 무엇인지 비교한다.

In [18]:
from sklearn.metrics.pairwise import cosine_similarity

sim_tdm   = pd.DataFrame(cosine_similarity(df_TDM.T),  index=doc_ids, columns=doc_ids)
sim_tfidf = pd.DataFrame(cosine_similarity(tf_idf.T), index=doc_ids, columns=doc_ids)

rows = []
for i, did in enumerate(doc_ids):
    row = {"문서": did, "카테고리": doc_cats[i]}
    for name, sim in [("TDM", sim_tdm), ("TF-IDF", sim_tfidf)]:
        s = sim.loc[did].drop(did)
        near = s.idxmax()
        row[f"{name} 최근접"] = f"{near} [{doc_cats[doc_ids.index(near)]}] {s.max():.3f}"
    rows.append(row)
pd.DataFrame(rows)

,문서,카테고리,TDM 최근접,TF-IDF 최근접
0,D00,경제,D01 [경제] 0.033,D01 [경제] 0.016
1,D01,경제,D09 [IT] 0.260,D09 [IT] 0.079
2,D02,경제,D08 [IT] 0.111,D08 [IT] 0.079
3,D03,사회,D06 [생활] 0.058,D06 [생활] 0.027
4,D04,사회,D02 [경제] 0.056,D02 [경제] 0.022
5,D05,생활,D02 [경제] 0.055,D02 [경제] 0.020
6,D06,생활,D03 [사회] 0.058,D03 [사회] 0.027
7,D07,IT,D09 [IT] 0.196,D09 [IT] 0.116
8,D08,IT,D09 [IT] 0.370,D09 [IT] 0.125
9,D09,IT,D08 [IT] 0.370,D08 [IT] 0.125


In [19]:
# D09(AI '지능 폭발' 경고)와 나머지 문서의 유사도 순위
BASE = "D09"
cmp = pd.DataFrame({"카테고리": doc_cats, "TDM": sim_tdm[BASE], "TF-IDF": sim_tfidf[BASE]},
                   index=doc_ids).drop(BASE)
cmp["TDM 순위"]    = cmp["TDM"].rank(ascending=False, method="min").astype(int)
cmp["TF-IDF 순위"] = cmp["TF-IDF"].rank(ascending=False, method="min").astype(int)
display(cmp.sort_values("TDM", ascending=False).round(3))


def shared_terms(mat, a, b, k=5):
    """코사인 유사도 = 정규화한 두 벡터의 단어별 곱의 합 -> 기여가 큰 공통 단어"""
    contrib = (mat[a] / np.linalg.norm(mat[a])) * (mat[b] / np.linalg.norm(mat[b]))
    return [f"{w}({v:.3f})" for w, v in contrib.nlargest(k).items() if v > 0]


for other in ["D01", "D07"]:
    print(f"\n{BASE}-{other} 유사도에 기여한 단어")
    print(f"   TDM    : {shared_terms(df_TDM.astype(float), BASE, other)}")
    print(f"   TF-IDF : {shared_terms(tf_idf, BASE, other)}")

,카테고리,TDM,TF-IDF,TDM 순위,TF-IDF 순위
D08,IT,0.370,0.125,1,1
D01,경제,0.260,0.079,2,3
D07,IT,0.196,0.116,3,2
D06,생활,0.023,0.013,4,4
D03,사회,0.013,0.008,5,5
D00,경제,0.010,0.006,6,6
D02,경제,0.006,0.003,7,7
D05,생활,0.006,0.001,8,8
D04,사회,0.002,0.000,9,9



D09-D01 유사도에 기여한 단어
   TDM    : ['AI(0.237)', '기술(0.008)', '기업(0.005)', '성능(0.004)', '규모(0.003)']
   TF-IDF : ['AI(0.063)', '기술(0.007)', '기업(0.004)', '성능(0.002)', '규모(0.001)']

D09-D07 유사도에 기여한 단어
   TDM    : ['AI(0.065)', '모델(0.026)', '오픈AI(0.020)', '앤트로픽(0.016)', '성능(0.013)']
   TF-IDF : ['모델(0.022)', 'AI(0.018)', '오픈AI(0.017)', '앤트로픽(0.014)', '작업(0.009)']


---

# 3. N-gram

BoW/TDM 은 단어 순서를 버린다. n-gram 은 인접한 n개 토큰을 하나의 피처로 만들어
국소적인 어순과 연어(collocation)를 담는다.

## 3.1 N-gram 생성

In [20]:
from nltk.util import ngrams as nltk_ngrams


def make_ngrams(tokens, n):
    """토큰 리스트에서 n개씩 묶어 n-gram 튜플 리스트를 만든다"""
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]


sample_tokens = doc_ls[6][:8]
print("원본 토큰 :", sample_tokens)
print("2-gram   :", make_ngrams(sample_tokens, 2)[:4])
print("3-gram   :", make_ngrams(sample_tokens, 3)[:4])

# nltk 로도 동일한 결과를 얻을 수 있다
print("\n[nltk.util.ngrams]")
print("2-gram   :", list(nltk_ngrams(sample_tokens, 2))[:4])
print("전체 문서에서 직접 구현과 동일 :",
      all(list(nltk_ngrams(t, n)) == make_ngrams(t, n) for t in doc_ls for n in (2, 3)))

원본 토큰 : ['췌장암', '진단자', '발견', '당시', '원격전이', '상대생존율', '소화불량', '체중']
2-gram   : [('췌장암', '진단자'), ('진단자', '발견'), ('발견', '당시'), ('당시', '원격전이')]
3-gram   : [('췌장암', '진단자', '발견'), ('진단자', '발견', '당시'), ('발견', '당시', '원격전이'), ('당시', '원격전이', '상대생존율')]

[nltk.util.ngrams]
2-gram   : [('췌장암', '진단자'), ('진단자', '발견'), ('발견', '당시'), ('당시', '원격전이')]
전체 문서에서 직접 구현과 동일 : True


## 3.2 N-gram TDM 생성 — 직접 구현

문서마다 1-gram + 2-gram + 3-gram 을 모두 모아 하나의 피처 목록으로 만든 뒤 빈도를 센다.
(n-gram 피처 이름은 토큰을 공백으로 이어 `체중 감소` 처럼 표기)

In [21]:
doc_ngrams = [[" ".join(g) for n in (1, 2, 3) for g in make_ngrams(tokens, n)]
              for tokens in doc_ls]

df_TDM_ngram = (pd.DataFrame([Counter(g) for g in doc_ngrams], index=doc_ids)
                .fillna(0).astype(int).T)
df_TDM_ngram.index.name = "n-gram"
print("1~3-gram 통합 TDM shape (n-gram x 문서) :", df_TDM_ngram.shape)

# 1-gram 을 제외하고 전체 빈도가 높은 n-gram
n_of = df_TDM_ngram.index.str.count(" ") + 1
multi = df_TDM_ngram[n_of >= 2]
multi.loc[multi.sum(axis=1).sort_values(ascending=False).index[:15]]

1~3-gram 통합 TDM shape (n-gram x 문서) : (4352, 10)


,D00,D01,D02,D03,D04,D05,D06,D07,D08,D09
n-gram,,,,,,,,,,
평균 월세,0,0,9,0,0,0,0,0,0,0
평균 전세보증금,0,0,7,0,0,0,0,0,0,0
GPT 솔,0,0,0,0,0,0,0,5,0,0
체중 감소,0,0,0,0,0,0,4,0,0,0
서울 평균,0,0,4,0,0,0,0,0,0,0
클로드 소넷,0,0,0,0,0,0,0,4,0,0
작업 비용,0,0,0,0,0,0,0,4,0,0
AI 개발,0,0,0,0,0,0,0,0,0,4
기술 발전,0,0,0,0,0,0,0,0,0,4


## 3.3 sklearn CountVectorizer 의 `ngram_range` 활용

1-gram / 2-gram / 3-gram / 1~3-gram 통합 TDM 을 만들고, 통합 TDM 이 직접 구현과 같은지 확인한다.

In [22]:
ngram_tdms = {}
for label, rng in [("1-gram", (1, 1)), ("2-gram", (2, 2)),
                   ("3-gram", (3, 3)), ("1~3-gram 통합", (1, 3))]:
    cv = CountVectorizer(lowercase=False, token_pattern=r"\S+", ngram_range=rng)
    m = cv.fit_transform(docs_joined)
    ngram_tdms[label] = (m, cv.get_feature_names_out())
    print(f"{label:>14} : {m.shape[1]:>5}개 피처")

m_all, terms_all = ngram_tdms["1~3-gram 통합"]
df_TDM_ngram_sk = pd.DataFrame(m_all.toarray().T, index=terms_all, columns=doc_ids)
df_TDM_ngram_sk.index.name = "n-gram"
print("\n1~3-gram 통합 TDM shape :", df_TDM_ngram_sk.shape)
print("직접 구현과 값이 동일    :", df_TDM_ngram_sk.sort_index().equals(df_TDM_ngram.sort_index()))
display(df_TDM_ngram_sk)

        1-gram :   917개 피처
        2-gram :  1659개 피처
        3-gram :  1776개 피처
   1~3-gram 통합 :  4352개 피처

1~3-gram 통합 TDM shape : (4352, 10)
직접 구현과 값이 동일    : True


,D00,D01,D02,D03,D04,D05,D06,D07,D08,D09
n-gram,,,,,,,,,,
AA,0,0,0,0,0,0,0,1,0,0
AA 오퍼스,0,0,0,0,0,0,0,1,0,0
AA 오퍼스 근접,0,0,0,0,0,0,0,1,0,0
AI,0,5,0,0,0,0,0,4,4,35
AI 개발,0,0,0,0,0,0,0,0,0,4
...,...,...,...,...,...,...,...,...,...,...
힌턴 토론토대,0,0,0,0,0,0,0,0,0,1
힌턴 토론토대 명예,0,0,0,0,0,0,0,0,0,1
힘쓰다,1,0,0,0,0,0,0,0,0,0


## 3.4 [실험] n 이 커질 때 무슨 일이 일어나는가

- **1회성(hapax) 비율** : 전체에서 딱 1번만 등장한 피처의 비율
- **2개 이상 문서 공유** : 두 개 이상의 문서에 나타나는 피처의 비율
  (= 문서 비교에 실제로 쓸 수 있는 피처의 비율)

In [23]:
rows = []
for label, (m, terms) in ngram_tdms.items():
    arr = m.toarray()
    n_feat = arr.shape[1]
    nnz = int(np.count_nonzero(arr))
    hapax  = int((arr.sum(axis=0) == 1).sum())            # 전체에서 딱 1번만 등장
    shared = int(((arr > 0).sum(axis=0) >= 2).sum())       # 2개 이상 문서에 등장
    rows.append({
        "n": label,
        "어휘(피처) 수": n_feat,
        "행렬 크기": f"{n_feat} x {len(doc_ids)}",
        "희소도(%)": round((1 - nnz / (n_feat * len(doc_ids))) * 100, 2),
        "1회성 비율(%)": round(hapax / n_feat * 100, 1),
        "2개 이상 문서 공유": shared,
        "공유 비율(%)": round(shared / n_feat * 100, 1),
    })
pd.DataFrame(rows)

,n,어휘(피처) 수,행렬 크기,희소도(%),1회성 비율(%),2개 이상 문서 공유,공유 비율(%)
0,1-gram,917,917 x 10,88.27,62.9,130,14.2
1,2-gram,1659,1659 x 10,89.96,93.7,7,0.4
2,3-gram,1776,1776 x 10,90.00,98.4,0,0.0
3,1~3-gram 통합,4352,4352 x 10,89.62,89.1,137,3.1


In [24]:
# 2개 이상 문서에 함께 등장한 2-gram / 3-gram 과 그 문서
for label in ["2-gram", "3-gram"]:
    m, terms = ngram_tdms[label]
    arr = m.toarray()
    shared_idx = np.where((arr > 0).sum(axis=0) >= 2)[0]
    print(f"[{label}] {len(shared_idx)}개")
    for j in shared_idx:
        docs = [f"{doc_ids[i]}({doc_cats[i]})" for i in np.where(arr[:, j] > 0)[0]]
        print(f"   {terms[j]:<10} : {', '.join(docs)}")

[2-gram] 7개
   AI 모델      : D07(IT), D09(IT)
   AI 에이전트    : D07(IT), D09(IT)
   GPT 아스트라   : D07(IT), D09(IT)
   글로벌 AI     : D01(경제), D08(IT)
   속도 빠르다     : D02(경제), D07(IT)
   오픈AI 출시    : D07(IT), D09(IT)
   작업 능력      : D07(IT), D09(IT)
[3-gram] 0개


---
# 4. 단어-문맥 동시등장 행렬 (Co-occurrence Matrix)

중심 단어 기준 좌우 `window_size` 만큼을 '문맥'으로 보고 함께 등장한 횟수를 센다.

## 4.1 동시등장 행렬 생성 함수

In [25]:
def build_cooccurrence(tokens, window_size=2, vocab_list=None):
    """
    tokens      : 한 문서의 토큰 리스트
    window_size : 중심 단어 좌우로 몇 칸까지를 문맥으로 볼지
    return      : (동시등장 행렬 DataFrame, 어휘 리스트)
    """
    if vocab_list is None:
        vocab_list = sorted(set(tokens))
    w2i = {w: i for i, w in enumerate(vocab_list)}

    n = len(vocab_list)
    co = np.zeros((n, n), dtype=int)

    for center_i, center in enumerate(tokens):
        if center not in w2i:
            continue
        start = max(0, center_i - window_size)
        end   = min(len(tokens), center_i + window_size + 1)
        for ctx_i in range(start, end):
            if ctx_i == center_i:            # 자기 자신 위치는 세지 않는다
                continue
            ctx = tokens[ctx_i]
            if ctx not in w2i:
                continue
            co[w2i[center], w2i[ctx]] += 1

    return pd.DataFrame(co, index=vocab_list, columns=vocab_list), vocab_list


# 작은 예제로 동작 확인
demo = "소화불량 체중 감소 황달 체중 감소 혈당 변화".split()
demo_co, _ = build_cooccurrence(demo, window_size=1)
print("예제 토큰 :", demo)
demo_co

예제 토큰 : ['소화불량', '체중', '감소', '황달', '체중', '감소', '혈당', '변화']


,감소,변화,소화불량,체중,혈당,황달
감소,0,0,0,2,1,1
변화,0,0,0,0,1,0
소화불량,0,0,0,1,0,0
체중,2,0,1,0,0,1
혈당,1,1,0,0,0,0
황달,1,0,0,1,0,0


## 4.2 실제 기사에 적용 — D06 (췌장암)

In [26]:
TARGET = 6                       # 분석할 문서 인덱스
target_tokens = doc_ls[TARGET]
print(f"대상 문서 : {doc_ids[TARGET]} | {doc_titles[TARGET]}")
print(f"토큰 수 {len(target_tokens)} / 고유 단어 {len(set(target_tokens))}")

WINDOW = 2
co_df, co_vocab = build_cooccurrence(target_tokens, window_size=WINDOW)
print(f"\n동시등장 행렬 shape : {co_df.shape}  (window_size={WINDOW})")
print("대칭 행렬 여부       :", (co_df.values == co_df.values.T).all())

# 전체를 다 찍으면 볼 수 없으므로 최빈 단어 12개만 잘라서 본다
top_words = [w for w, _ in Counter(target_tokens).most_common(12)]
co_df.loc[top_words, top_words]

대상 문서 : D06 | 소화불량인 줄 알았는데…췌장암 43%는 발견 때 이미 전이, 생존율 2.4%
토큰 수 350 / 고유 단어 175

동시등장 행렬 shape : (175, 175)  (window_size=2)
대칭 행렬 여부       : True


,췌장암,황달,췌장,필요,수술,발견,체중,검사,증상,혈당,암,변화
췌장암,0,0,0,4,3,3,0,1,1,2,1,2
황달,0,0,0,1,0,0,4,0,0,4,0,2
췌장,0,0,0,1,0,0,0,2,0,0,2,0
필요,4,1,1,2,0,0,0,2,0,0,0,0
수술,3,0,0,0,4,0,0,0,0,0,0,0
발견,3,0,0,0,0,0,0,0,0,0,2,0
체중,0,4,0,0,0,0,0,0,1,1,0,0
검사,1,0,2,2,0,0,0,0,0,0,1,0
증상,1,0,0,0,0,0,1,0,2,0,0,0
혈당,2,4,0,0,0,0,1,0,0,0,0,4


## 4.3 특정 단어와 함께 등장한 단어 (문맥)

In [27]:
for word in ["췌장암", "황달", "체중", "복부초음파"]:
    top_ctx = co_df.loc[word].sort_values(ascending=False).head(6)
    top_ctx = top_ctx[top_ctx > 0]
    print(f"\n'{word}' 의 문맥 단어 (window={WINDOW})")
    print("  " + ", ".join(f"{w}({c})" for w, c in top_ctx.items()))


'췌장암' 의 문맥 단어 (window=2)
  필요(4), 수술(3), 의심(3), 복부초음파(3), 환자(3), 발견(3)

'황달' 의 문맥 단어 (window=2)
  체중(4), 혈당(4), 감소(3), 변화(2), 복부(2), 통증(2)

'체중' 의 문맥 단어 (window=2)
  감소(4), 황달(4), 계속(2), 소화불량(2), 이유(2), 혈당(1)

'복부초음파' 의 문맥 단어 (window=2)
  췌장암(3), 작다(3), 놓치다(2), 필요(2), 발견(1), 상담(1)


## 4.4 [실험] window_size 에 따른 변화

In [28]:
KEY = "췌장암"
rows = []
for w in [1, 2, 5, 10]:
    cdf, _ = build_cooccurrence(target_tokens, window_size=w)
    arr = cdf.values
    n = arr.shape[0]
    nnz = int(np.count_nonzero(arr))
    top5 = cdf.loc[KEY].sort_values(ascending=False).head(5)
    rows.append({
        "window_size": w,
        "0이 아닌 칸": nnz,
        "밀도(%)": round(nnz / (n * n) * 100, 2),
        "총 카운트 합": int(arr.sum()),
        f"'{KEY}'의 이웃 top5": ", ".join(f"{t}({c})" for t, c in top5.items()),
    })
pd.DataFrame(rows)

,window_size,0이 아닌 칸,밀도(%),총 카운트 합,'췌장암'의 이웃 top5
0,1,617,2.01,698,"의심(3), 환자(2), 진단자(2), 작다(2), 수술(2)"
1,2,1212,3.96,1394,"필요(4), 수술(3), 의심(3), 복부초음파(3), 환자(3)"
2,5,2760,9.01,3470,"필요(7), 발견(6), 의심(6), 당뇨병(5), 검사(5)"
3,10,4957,16.19,6890,"췌장암(14), 검사(9), 수술(8), 필요(8), 증상(8)"


## 4.5 [실험] 문장 경계를 넘는 문맥

윈도는 문장이 끝나도 멈추지 않는다. 문장 단위로 잘라서 각 문장 안에서만 윈도를 적용한 결과와 비교해,
전체 카운트 중 **문장 경계를 넘어가서 센 비율**을 확인한다.

In [29]:
sentences = re.split(r"(?<=[.?!])\s+", clean_text(NEWS[TARGET]["text"]))
sent_tokens = [preprocess(s) for s in sentences]
print(f"문장 수 : {len(sentences)}")
print("문장 단위 토큰을 이어 붙이면 문서 토큰과 동일 :", sum(sent_tokens, []) == target_tokens)

rows = []
for w in [1, 2, 5, 10]:
    whole, _ = build_cooccurrence(target_tokens, window_size=w, vocab_list=co_vocab)
    in_sent = sum(build_cooccurrence(t, window_size=w, vocab_list=co_vocab)[0].values for t in sent_tokens)
    total, inside = int(whole.values.sum()), int(in_sent.sum())
    rows.append({"window_size": w, "문서 전체 카운트": total, "문장 안 카운트": inside,
                 "문장 경계를 넘은 비율(%)": round((total - inside) / total * 100, 1)})
pd.DataFrame(rows)

문장 수 : 40
문장 단위 토큰을 이어 붙이면 문서 토큰과 동일 : True


,window_size,문서 전체 카운트,문장 안 카운트,문장 경계를 넘은 비율(%)
0,1,698,620,11.2
1,2,1394,1160,16.8
2,5,3470,2332,32.8
3,10,6890,3272,52.5
